# ABS Personal Income by Remoteness Area (6524.0.55.002)

Employee income by ABS Remoteness Area, constructed from:

- **Table 5.4** of *Personal Income in Australia* (cat. 6524.0.55.002) — employee income by SA2.
- **ASGS Edition 3** SA1 → Remoteness Area allocation file, rolled up to SA2 by modal RA.

RA mean income = total employee income in RA / total earners in RA (i.e. an earner-weighted mean across SA2s).

## Set-up

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
from readabs.download_cache import get_file
from readabs.get_abs_links import get_abs_links
from mgplot import bar_plot_finalise, line_plot_finalise, set_chart_dir, clear_chart_dir

In [2]:
INCOME_URL = (
    'https://www.abs.gov.au/statistics/labour/earnings-and-working-conditions/'
    'personal-income-australia/latest-release'
)
ASGS_RA_URL = (
    'https://www.abs.gov.au/statistics/standards/'
    'australian-statistical-geography-standard-asgs-edition-3/'
    'jul2021-jun2026/access-and-downloads/allocation-files'
)
CHART_DIR = './CHARTS/6524.0.55.002 - Personal Income by Remoteness/'
Path(CHART_DIR).mkdir(parents=True, exist_ok=True)
set_chart_dir(CHART_DIR)
clear_chart_dir()

SHOW = False
YEAR = '2022-23'
RA_ORDER = [
    'Major Cities of Australia',
    'Inner Regional Australia',
    'Outer Regional Australia',
    'Remote Australia',
    'Very Remote Australia',
]
RA_LABELS = {
    'Major Cities of Australia': 'Major Cities',
    'Inner Regional Australia': 'Inner Regional',
    'Outer Regional Australia': 'Outer Regional',
    'Remote Australia': 'Remote',
    'Very Remote Australia': 'Very Remote',
}

## Build SA2 → Remoteness mapping

SA1 codes are 11 digits; the first 9 are the SA2 code. Each SA2 is assigned its modal RA (the RA containing the most SA1s in that SA2).

In [3]:
def fetch_abs_xlsx(landing_url: str, filename_match) -> bytes:
    """Download an ABS .xlsx from a landing page (cached by readabs).

    filename_match is applied to each decoded basename; the first match wins.
    Resolving via the landing page keeps this working as ABS moves files into
    new dated folders, so the notebook needs no manually-downloaded local copy.
    """
    links = get_abs_links(landing_url)
    url = next(
        link for link in links.get('.xlsx', [])
        if filename_match(unquote(link.rsplit('/', 1)[-1]))
    )
    return get_file(url)


def load_sa2_to_ra() -> pd.Series:
    """Return Series indexed by SA2 code with the modal Remoteness Area name."""
    raw = fetch_abs_xlsx(ASGS_RA_URL, lambda name: name == 'RA_2021_AUST.xlsx')
    df = pd.read_excel(BytesIO(raw), sheet_name='SA1_RA_2021_AUST')
    df['SA2'] = df['SA1_CODE_2021'].astype(str).str[:9]
    return df.groupby('SA2')['RA_NAME_2021'].agg(lambda x: x.mode().iloc[0])


sa2_ra = load_sa2_to_ra()
print(f'SA2 → RA mappings: {len(sa2_ra)}')

SA2 → RA mappings: 2473


## Load employee income by SA2

In [4]:
def load_sa2_income() -> pd.DataFrame:
    """Load Table 5.4: employee income by SA2 across 2018-19 to 2022-23."""
    raw = fetch_abs_xlsx(
        INCOME_URL, lambda name: name.startswith('Table 5 - Employee income'),
    )
    df = pd.read_excel(BytesIO(raw), sheet_name='Table 5.4', header=[5, 6])
    df.columns = [
        f'{a}_{b}' if 'Unnamed' not in a else b for a, b in df.columns
    ]
    df = df.rename(columns={'SA2': 'SA2_CODE', 'SA2 NAME': 'SA2_NAME'})
    # keep only 9-digit SA2 codes (strip state/national aggregate rows)
    df = df[df['SA2_CODE'].astype(str).str.match(r'^\d{9}$', na=False)].copy()
    df['SA2_CODE'] = df['SA2_CODE'].astype(str)
    # 'np' = not published; coerce to NaN
    years = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
    for yr in years:
        for col in [f'Earners (persons)_{yr}', f'Sum ($)_{yr}']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


income = load_sa2_income()
print(f'SA2 income rows: {len(income)}')

SA2 income rows: 2450


## Aggregate to Remoteness Area

In [5]:
def aggregate_by_ra(income: pd.DataFrame, sa2_ra: pd.Series, year: str) -> pd.Series:
    """Earner-weighted mean employee income by Remoteness Area for the given year."""
    merged = income.merge(
        sa2_ra.rename('RA'), left_on='SA2_CODE', right_index=True, how='left',
    )
    merged = merged[merged['RA'].isin(RA_ORDER)]
    sums = merged.groupby('RA')[f'Sum ($)_{year}'].sum()
    earners = merged.groupby('RA')[f'Earners (persons)_{year}'].sum()
    mean = (sums / earners).reindex(RA_ORDER)
    mean.index = [RA_LABELS[r] for r in mean.index]
    return mean.rename('Mean income ($)')


mean_by_ra = aggregate_by_ra(income, sa2_ra, YEAR)
print(mean_by_ra.round(0))

Major Cities      77820.0
Inner Regional    65506.0
Outer Regional    64140.0
Remote            72642.0
Very Remote       64370.0
Name: Mean income ($), dtype: float64


## Plot

In [6]:
def plot_bar(s: pd.Series, year: str) -> None:
    bar_plot_finalise(
        s.to_frame(),
        title=f'Mean Employee Income by Remoteness Area: {year}',
        ylabel='Mean income ($ per earner per year)',
        annotate=True,
        rounding=0,
        color=['steelblue'],
        legend=False,
        rfooter='ABS 6524.0.55.002 T5.4; ASGS Ed.3 RA',
        lfooter=(
            'Australia. Earner-weighted mean across SA2s. '
            'SA2 assigned to modal Remoteness Area. '
        ),
        show=SHOW,
    )


plot_bar(mean_by_ra, YEAR)

## Time series in nominal $

Earner-weighted mean employee income by Remoteness Area, 2018-19 to 2022-23.

In [ ]:
YEARS = ["2018-19", "2019-20", "2020-21", "2021-22", "2022-23"]


def time_series_by_ra(income: pd.DataFrame, sa2_ra: pd.Series) -> pd.DataFrame:
    """DataFrame: rows are financial years (PeriodIndex, Y-JUN), columns are RAs."""
    data = pd.DataFrame({yr: aggregate_by_ra(income, sa2_ra, yr) for yr in YEARS}).T
    end_years = [int(f"20{yr.split('-')[1]}") for yr in data.index]
    data.index = pd.PeriodIndex(
        [pd.Period(year=y, freq="Y-JUN") for y in end_years], name="year",
    )
    return data


def plot_lines(df: pd.DataFrame) -> None:
    line_plot_finalise(
        df,
        title="Mean Employee Income by Remoteness Area",
        ylabel="$ per earner per year",
        xlabel="Financial year ending June",
        width=2,
        annotate=True,
        rounding=0,
        xlim=(df.index[0], df.index[-1]),
        rfooter="ABS 6524.0.55.002 T5.4; ASGS Ed.3 RA",
        lfooter=(
            "Australia. Earner-weighted mean across SA2s. "
            "SA2 assigned to modal Remoteness Area. "
        ),
        show=SHOW,
    )


ts = time_series_by_ra(income, sa2_ra)
print(ts.round(0))
plot_lines(ts)


## Time series indexed to 2018-19 = 100

Same series rebased so growth rates across remoteness areas are directly comparable.

In [ ]:
def plot_lines_indexed(df: pd.DataFrame) -> None:
    indexed = df.div(df.iloc[0]) * 100
    line_plot_finalise(
        indexed,
        title="Mean Employee Income by Remoteness Area: Indexed",
        ylabel="Index (2018-19 = 100)",
        xlabel="Financial year ending June",
        width=2,
        annotate=True,
        rounding=1,
        xticks=list(indexed.index),
        xlim=(indexed.index[0], indexed.index[-1]),
        rfooter="ABS 6524.0.55.002 T5.4; ASGS Ed.3 RA",
        lfooter=(
            "Australia. Earner-weighted mean across SA2s. "
            "SA2 assigned to modal Remoteness Area. "
        ),
        show=SHOW,
    )


plot_lines_indexed(ts)


## Watermark

In [9]:
%load_ext watermark
%watermark -u -t -d --iversions

Last updated: 2026-05-11 20:02:56

mgplot : 0.2.23
pandas : 3.0.2
pathlib: 1.0.1

